Face web scraping do forecasts da pagina  https://www.forecasts.org/all-forecasts.htm

gera df con os títulos ao consultar (symbol , url)

In [77]:
import sqlite3
import pandas as pd

# Conectar ao banco de dados SQLite
conn = sqlite3.connect("sqforecasts.db")

# Criar a consulta SQL
query = "SELECT id_titulos, symbol, url FROM tbtitulos"

# Criar o DataFrame com os dados da consulta
dftitulos = pd.read_sql(query, conn)

# Fechar a conexão
conn.close()

# Exibir o DataFrame
display(dftitulos)

,id_titulos,symbol,url
0,1,DXY,https://www.forecasts.org/dollar-forecast.htm
1,2,US10Y,https://www.forecasts.org/10yrT.htm
2,3,USIR,https://www.forecasts.org/inflation.htm
3,4,SPX,https://www.forecasts.org/stpoor.htm
4,5,EUR/USD,https://www.forecasts.org/euro.htm


In [79]:
# Selecionando um id específico
id_especifico = 2  # Alterar para o ID desejado

In [81]:
import pandas as pd
dftituloselec = dftitulos[dftitulos["id_titulos"] == id_especifico]
dftituloselec.columns = dftituloselec.columns.str.strip()  # Remove espaços nos nomes das colunas
# Definir a url 
url=(dftituloselec['url'].iloc[0]) #define a url para a função get_fore_dados(url)
id_titulos= int(dftituloselec['id_titulos'].iloc[0])
print(dftituloselec)
print(url)
print(id_titulos)

   id_titulos symbol                                  url
1           2  US10Y  https://www.forecasts.org/10yrT.htm
https://www.forecasts.org/10yrT.htm
2


Extrae com a url selecionada a tabula com os dados de forecast com a função
get_fore_dados(url) return: df_forecasts_scrap, updated_date

In [84]:
from forefunc import get_fore_dados 

# Chamar a função para obter os dados
df_forecasts_scrap, last_updated = get_fore_dados(url)

In [85]:

# Exibir os dados
display(f"idtitulos:{id_titulos}")
display(f"Última atualização: {last_updated}")
display(df_forecasts_scrap.columns)
display(df_forecasts_scrap)

'idtitulos:2'

'Última atualização: 2025-05-15 00:00:00'

Index(['Month', 'Date', 'Forecast Value', 'Avg Error'], dtype='object')

,Month,Date,Forecast Value,Avg Error
0,Apr 2025,2025-04-01,4.28,0.000
1,May 2025,2025-05-01,4.40,0.078
2,Jun 2025,2025-06-01,4.29,0.100
3,Jul 2025,2025-07-01,4.04,0.110
4,Aug 2025,2025-08-01,3.78,0.120
5,Sep 2025,2025-09-01,3.16,0.130
6,Oct 2025,2025-10-01,2.97,0.140
7,Nov 2025,2025-11-01,2.99,0.140
8,Dec 2025,2025-12-01,2.64,0.150


Insere na tabela tbforecasts (si no existe registro) um novo e devolve os valores: existsforecast e
id_forecastins com a função insere_forecast(id_titulos, data) return id_forecastins, existsforecast

In [125]:
import importlib
import forefunc
importlib.reload(forefunc)
from forefunc  import insere_forecast

id_forecastins, existsforecast = insere_forecast(id_titulos,last_updated)


In [128]:
if existsforecast ==0:
    print (f'existsforecast:{ existsforecast} registro inserido')
    print (f'id_forecastins:{id_forecastins} ultimo id')
else:
    print (f'existsforecast:{ existsforecast} el registro ja existe')
    print (f'id_forecastins:{id_forecastins} no se gero id')


existsforecast:1 el registro ja existe
id_forecastins:0 no se gero id


Si existforecast = 0 Crea o registro para inserir em na tabela tbtituloshist
tituloshist_reg(df_forecast_scrap)
return dftituloshistins


In [135]:
import pandas as pd

# Valores do DataFrame original
primeiro_date = str(df_forecasts_scrap["Date"].iloc[0])  # Pegando o primeiro valor de Date
primeiro_forecast_value = float(df_forecasts_scrap["Forecast Value"].iloc[0])  # Pegando o primeiro valor de Forecast Value

# Criando o novo DataFrame
dftituloshistins = pd.DataFrame({
    "id_titulos": [id_titulos],
    "data": [primeiro_date],
    "valor": [primeiro_forecast_value]
})

display(dftituloshistins)

,id_titulos,data,valor
0,2,2025-04-01 00:00:00,4.28


Insere registro e verifica que no este duplicado
forecasthist_insere (dftituloshistins)
return existshist

In [138]:
import importlib
import forefunc
importlib.reload(forefunc)
from forefunc import forecasthist_insere
    
existshist = forecasthist_insere(dftituloshistins)


In [140]:
print (f'existshist:{ existshist}')

existshist:1


creo dfforecastdadosins para inserir os dados na tabela dfforecastdados solo si existforecast = 0

In [146]:
# Criando o novo DataFrame
dfforecastsdados = pd.DataFrame({
    
    "id_forecasts":  [int(id_forecastins)] * len(df_forecasts_scrap),
    "data": df_forecasts_scrap['Date'], 
    "valor": df_forecasts_scrap['Forecast Value'],
    "error": df_forecasts_scrap['Avg Error'] ,
})
display (dfforecastsdados)

,id_forecasts,data,valor,error
0,0,2025-04-01,4.28,0.000
1,0,2025-05-01,4.40,0.078
2,0,2025-06-01,4.29,0.100
3,0,2025-07-01,4.04,0.110
4,0,2025-08-01,3.78,0.120
5,0,2025-09-01,3.16,0.130
6,0,2025-10-01,2.97,0.140
7,0,2025-11-01,2.99,0.140
8,0,2025-12-01,2.64,0.150


In [148]:
import importlib
import forefunc
importlib.reload(forefunc)
from forefunc import forecastsdados_insere


existsforecastsdados = forecastsdados_insere (dfforecastsdados)

In [150]:
if existsforecastsdados == 0 :
    print("Dados inseridos em tbforecastsdados com sucesso!")
else:
    print("Nenhum dado novo para inserir em tbforecastsdados.")

Nenhum dado novo para inserir em tbforecastsdados.
